# Value Stock Screener
# Screens for stocks with:
# - Price: $10-$100
# - Float < 1M shares
# - P/E Ratio < 10
# - P/B Ratio < 1.0

In [ ]:
# Initialize QuantBook
from datetime import datetime, timedelta

qb = QuantBook()

# Set the date for screening (use a recent business day)
# Note: January 1, 2024 was a holiday. Use January 2nd instead
screening_date = datetime(2024, 1, 2)
qb.set_start_date(screening_date)

print(f"Screening date set to: {screening_date}")

In [ ]:
# Get S&P 500 constituent symbols using universe history
# This gets the list of stocks that were in the SPY ETF on the screening date
print(f"Fetching universe data for date: {screening_date}")

universe_history = qb.universe_history(
    qb.universe.etf('SPY'),
    screening_date - timedelta(1),
    screening_date
)

print(f"Universe history shape: {universe_history.shape}")
print(f"Universe history empty: {universe_history.empty}")

if universe_history.empty:
    print("\nTrying alternative date range...")
    # Try a wider date range
    universe_history = qb.universe_history(
        qb.universe.etf('SPY'),
        screening_date - timedelta(5),
        screening_date + timedelta(1)
    )
    print(f"New universe history shape: {universe_history.shape}")

if not universe_history.empty:
    # Extract symbols from the universe history
    spy_constituents = [fundamental.symbol for fundamental in universe_history.iloc[0]]
    print(f"Found {len(spy_constituents)} symbols in SPY")
else:
    print("ERROR: Could not fetch universe data. Please check the date range.")
    spy_constituents = []

In [ ]:
# Screen stocks based on criteria (FIXED VERSION)
screened_stocks = []

if not universe_history.empty and len(spy_constituents) > 0:
    print(f"Screening {len(spy_constituents)} stocks...")
    
    for symbol in spy_constituents:
        try:
            # Add the equity to access its fundamental data
            equity = qb.add_equity(symbol, Resolution.DAILY)
            
            # Get current price
            history = qb.history(symbol, 1, Resolution.DAILY)
            if history.empty:
                continue
            
            price = history['close'].iloc[-1]
            
            # Check price range ($10-$100)
            if price < 10 or price > 100:
                continue
            
            # Get fundamentals from the security object
            if symbol not in qb.securities:
                continue
                
            security = qb.securities[symbol]
            
            # Check if fundamentals exist
            if not hasattr(security, 'fundamentals') or security.fundamentals is None:
                continue
            
            fundamentals = security.fundamentals
            
            # Get valuation ratios and market cap
            pe_ratio = fundamentals.valuation_ratios.pe_ratio
            pb_ratio = fundamentals.valuation_ratios.pb_ratio
            market_cap = fundamentals.market_cap
            
            # Validate we have the data we need
            if pe_ratio is None or pb_ratio is None or market_cap is None:
                continue
            
            if market_cap <= 0 or price <= 0:
                continue
            
            # Get shares outstanding (used for float estimation)
            shares_outstanding = market_cap / price
            
            # Note: QuantConnect doesn't directly provide float shares
            # We use shares outstanding as an approximation
            # In reality, float is typically lower than shares outstanding
            
            # Apply screening criteria
            if (pe_ratio > 0 and pe_ratio < 10 and
                pb_ratio > 0 and pb_ratio < 1.0 and
                shares_outstanding > 0 and shares_outstanding < 1_000_000):
                
                screened_stocks.append({
                    'Symbol': str(symbol),
                    'Price': round(price, 2),
                    'P/E Ratio': round(pe_ratio, 2),
                    'P/B Ratio': round(pb_ratio, 2),
                    'Shares (M)': round(shares_outstanding / 1_000_000, 2),
                    'Market Cap ($M)': round(market_cap / 1_000_000, 2)
                })
                
        except Exception as e:
            # Uncomment to see detailed errors
            # print(f"Error processing {symbol}: {str(e)}")
            continue
    
    print(f"\nScreening complete. Found {len(screened_stocks)} stocks matching criteria.")
else:
    print("No universe data available to screen.")

In [ ]:
# Display results as a DataFrame
import pandas as pd

if screened_stocks:
    results_df = pd.DataFrame(screened_stocks)
    results_df = results_df.sort_values('P/E Ratio')
    display(results_df)
    
    # Summary statistics
    print("\n=== Summary Statistics ===")
    print(f"Total stocks found: {len(results_df)}")
    print(f"\nAverage P/E: {results_df['P/E Ratio'].mean():.2f}")
    print(f"Average P/B: {results_df['P/B Ratio'].mean():.2f}")
    print(f"Average Price: ${results_df['Price'].mean():.2f}")
    print(f"Average Shares Outstanding: {results_df['Shares (M)'].mean():.2f}M shares")
else:
    print("No stocks found matching all criteria.")
    print("\nConsider relaxing some constraints:")
    print("- Increase P/E threshold (currently < 10)")
    print("- Increase P/B threshold (currently < 1.0)")
    print("- Increase shares outstanding limit (currently < 1M)")
    print("  Note: Very few publicly traded stocks have < 1M shares")
    print("  Consider trying < 10M or < 50M for more realistic results")
    print("- Expand universe beyond S&P 500 to include small-caps (IWM)")

In [ ]:
# Diagnostic: Check what values we're actually seeing
# Fixed to properly access fundamental data

diagnostic_data = []
sample_count = 0
max_samples = 20  # Show first 20 stocks
error_log = []

if not universe_history.empty and len(spy_constituents) > 0:
    print(f"Examining first {max_samples} stocks to see actual values...\n")
    
    for idx, symbol in enumerate(spy_constituents):
        if sample_count >= max_samples:
            break
            
        try:
            print(f"Processing {idx+1}: {symbol}... ", end="")
            
            # Add the equity to access its fundamental data
            equity = qb.add_equity(symbol, Resolution.DAILY)
            
            # Get current price
            history = qb.history(symbol, 1, Resolution.DAILY)
            if history.empty:
                print("❌ No price history")
                error_log.append(f"{symbol}: No price history")
                continue
            
            price = history['close'].iloc[-1]
            print(f"Price: ${price:.2f}, ", end="")
            
            # Get fundamentals from the security object
            if symbol not in qb.securities:
                print("❌ Security not found")
                error_log.append(f"{symbol}: Security not found")
                continue
                
            security = qb.securities[symbol]
            
            # Check if fundamentals exist
            if not hasattr(security, 'fundamentals') or security.fundamentals is None:
                print("❌ No fundamentals")
                error_log.append(f"{symbol}: No fundamentals")
                continue
            
            fundamentals = security.fundamentals
            
            # Get valuation ratios and market cap
            pe_ratio = fundamentals.valuation_ratios.pe_ratio
            pb_ratio = fundamentals.valuation_ratios.pb_ratio
            market_cap = fundamentals.market_cap
            
            print(f"P/E: {pe_ratio}, P/B: {pb_ratio}, MCap: ${market_cap/1e9:.2f}B")
            
            if market_cap is None or price is None or price <= 0 or market_cap <= 0:
                print(f"  ⚠️ Invalid data")
                continue
            
            shares_outstanding = market_cap / price
            
            diagnostic_data.append({
                'Symbol': str(symbol),
                'Price': round(price, 2),
                'P/E': round(pe_ratio, 2) if pe_ratio and pe_ratio > 0 else None,
                'P/B': round(pb_ratio, 2) if pb_ratio and pb_ratio > 0 else None,
                'Shares (M)': round(shares_outstanding / 1_000_000, 2),
                'Market Cap ($B)': round(market_cap / 1_000_000_000, 2),
                'Meets Price': '✓' if 10 <= price <= 100 else '✗',
                'Meets P/E': '✓' if pe_ratio and 0 < pe_ratio < 10 else '✗',
                'Meets P/B': '✓' if pb_ratio and 0 < pb_ratio < 1.0 else '✗',
                'Meets Shares': '✓' if shares_outstanding < 1_000_000 else '✗'
            })
            
            sample_count += 1
            
        except Exception as e:
            print(f"❌ Error: {e}")
            error_log.append(f"{symbol}: {e}")
            continue
    
    print(f"\n\n{'='*60}")
    print(f"Successfully processed: {len(diagnostic_data)} stocks")
    print(f"Errors encountered: {len(error_log)}")
    
    if error_log and len(error_log) <= 10:
        print("\nErrors:")
        for err in error_log:
            print(f"  - {err}")
    
    if diagnostic_data:
        diag_df = pd.DataFrame(diagnostic_data)
        print("\nSample of stocks with their actual values:\n")
        display(diag_df)
        
        # Show statistics
        print("\n=== Statistics from Sample ===")
        print(f"Price range: ${diag_df['Price'].min():.2f} - ${diag_df['Price'].max():.2f}")
        print(f"P/E range: {diag_df['P/E'].min():.2f} - {diag_df['P/E'].max():.2f}")
        print(f"P/B range: {diag_df['P/B'].min():.2f} - {diag_df['P/B'].max():.2f}")
        print(f"Shares outstanding range: {diag_df['Shares (M)'].min():.2f}M - {diag_df['Shares (M)'].max():.2f}M")
        print(f"\nStocks meeting each criterion:")
        print(f"  Price $10-$100: {(diag_df['Meets Price'] == '✓').sum()} / {len(diag_df)}")
        print(f"  P/E < 10: {(diag_df['Meets P/E'] == '✓').sum()} / {len(diag_df)}")
        print(f"  P/B < 1.0: {(diag_df['Meets P/B'] == '✓').sum()} / {len(diag_df)}")
        print(f"  Shares < 1M: {(diag_df['Meets Shares'] == '✓').sum()} / {len(diag_df)}")
    else:
        print("\n⚠️ No valid data could be extracted from any stocks!")
        print("This suggests an issue with fundamental data access.")
else:
    print("No universe data available for diagnostics.")

## Diagnostic: View Actual Stock Data
Let's see what the actual values look like to understand why no stocks match

In [ ]:
# Relaxed screener with more realistic thresholds (FIXED VERSION)
# Adjust these values based on the diagnostic output above

# CONFIGURABLE PARAMETERS
MIN_PRICE = 10
MAX_PRICE = 100
MAX_PE = 15  # Relaxed from 10
MAX_PB = 1.5  # Relaxed from 1.0
MAX_SHARES_MILLION = 50  # Relaxed from 1M to 50M shares (more realistic)

screened_relaxed = []

if not universe_history.empty and len(spy_constituents) > 0:
    print(f"Screening with relaxed criteria:")
    print(f"  Price: ${MIN_PRICE}-${MAX_PRICE}")
    print(f"  P/E < {MAX_PE}")
    print(f"  P/B < {MAX_PB}")
    print(f"  Shares < {MAX_SHARES_MILLION}M")
    print(f"\nProcessing {len(spy_constituents)} stocks...\n")
    
    for symbol in spy_constituents:
        try:
            # Add the equity to access its fundamental data
            equity = qb.add_equity(symbol, Resolution.DAILY)
            
            # Get current price
            history = qb.history(symbol, 1, Resolution.DAILY)
            if history.empty:
                continue
            
            price = history['close'].iloc[-1]
            
            # Check price range
            if price < MIN_PRICE or price > MAX_PRICE:
                continue
            
            # Get fundamentals from the security object
            if symbol not in qb.securities:
                continue
                
            security = qb.securities[symbol]
            
            # Check if fundamentals exist
            if not hasattr(security, 'fundamentals') or security.fundamentals is None:
                continue
            
            fundamentals = security.fundamentals
            
            # Get valuation ratios and market cap
            pe_ratio = fundamentals.valuation_ratios.pe_ratio
            pb_ratio = fundamentals.valuation_ratios.pb_ratio
            market_cap = fundamentals.market_cap
            
            # Validate we have the data we need
            if pe_ratio is None or pb_ratio is None or market_cap is None:
                continue
            
            if market_cap <= 0 or price <= 0:
                continue
            
            # Get shares outstanding
            shares_outstanding = market_cap / price
            shares_in_millions = shares_outstanding / 1_000_000
            
            # Apply screening criteria (relaxed)
            if (pe_ratio > 0 and pe_ratio < MAX_PE and
                pb_ratio > 0 and pb_ratio < MAX_PB and
                shares_in_millions < MAX_SHARES_MILLION):
                
                screened_relaxed.append({
                    'Symbol': str(symbol),
                    'Price': round(price, 2),
                    'P/E Ratio': round(pe_ratio, 2),
                    'P/B Ratio': round(pb_ratio, 2),
                    'Shares (M)': round(shares_in_millions, 2),
                    'Market Cap ($M)': round(market_cap / 1_000_000, 2)
                })
                
        except Exception as e:
            continue
    
    print(f"Found {len(screened_relaxed)} stocks matching relaxed criteria.")
    
    if screened_relaxed:
        relaxed_df = pd.DataFrame(screened_relaxed)
        relaxed_df = relaxed_df.sort_values('P/E Ratio')
        display(relaxed_df)
        
        # Summary statistics
        print("\n=== Summary Statistics ===")
        print(f"Total stocks found: {len(relaxed_df)}")
        print(f"Average P/E: {relaxed_df['P/E Ratio'].mean():.2f}")
        print(f"Average P/B: {relaxed_df['P/B Ratio'].mean():.2f}")
        print(f"Average Price: ${relaxed_df['Price'].mean():.2f}")
        print(f"Average Shares: {relaxed_df['Shares (M)'].mean():.2f}M")
    else:
        print("\nNo stocks found even with relaxed criteria.")
        print("Consider further relaxing the constraints or expanding the universe.")
else:
    print("No universe data available.")

In [ ]:
# Industry-based ranking scanner
# Collects all stocks with P/E, P/B, ROE and ranks them within each industry

import numpy as np

all_stocks_data = []

if not universe_history.empty and len(spy_constituents) > 0:
    print(f"Collecting data from {len(spy_constituents)} stocks...")
    
    processed = 0
    for symbol in spy_constituents:
        try:
            # Add the equity
            equity = qb.add_equity(symbol, Resolution.DAILY)
            
            # Get price
            history = qb.history(symbol, 1, Resolution.DAILY)
            if history.empty:
                continue
            
            price = history['close'].iloc[-1]
            
            # Get fundamentals
            if symbol not in qb.securities:
                continue
            
            security = qb.securities[symbol]
            
            if not hasattr(security, 'fundamentals') or security.fundamentals is None:
                continue
            
            fundamentals = security.fundamentals
            
            # Get valuation ratios
            pe_ratio = fundamentals.valuation_ratios.pe_ratio
            pb_ratio = fundamentals.valuation_ratios.pb_ratio
            
            # Get ROE
            roe = fundamentals.operation_ratios.roe.value if fundamentals.operation_ratios.roe else None
            
            # Get industry and sector
            asset_classification = fundamentals.asset_classification
            industry = asset_classification.morningstar_industry_group_code if asset_classification else "Unknown"
            sector = asset_classification.morningstar_sector_code if asset_classification else "Unknown"
            
            # Get market cap
            market_cap = fundamentals.market_cap
            
            # Skip if missing critical data
            if pe_ratio is None or pb_ratio is None or roe is None:
                continue
            
            if pe_ratio <= 0 or pb_ratio <= 0 or market_cap is None or market_cap <= 0:
                continue
            
            all_stocks_data.append({
                'Symbol': str(symbol),
                'Price': price,
                'P/E': pe_ratio,
                'P/B': pb_ratio,
                'ROE': roe * 100,  # Convert to percentage
                'Industry': industry,
                'Sector': sector,
                'Market Cap ($B)': market_cap / 1e9
            })
            
            processed += 1
            if processed % 50 == 0:
                print(f"  Processed {processed} stocks...")
            
        except Exception as e:
            continue
    
    print(f"\n✓ Successfully collected data from {len(all_stocks_data)} stocks")
    
    if all_stocks_data:
        stocks_df = pd.DataFrame(all_stocks_data)
        
        # Calculate composite score
        # Lower P/E and P/B is better, higher ROE is better
        # Normalize each metric to 0-100 scale
        
        # For P/E and P/B: invert so lower values get higher scores
        stocks_df['PE_Score'] = 100 * (1 - (stocks_df['P/E'] - stocks_df['P/E'].min()) / (stocks_df['P/E'].max() - stocks_df['P/E'].min()))
        stocks_df['PB_Score'] = 100 * (1 - (stocks_df['P/B'] - stocks_df['P/B'].min()) / (stocks_df['P/B'].max() - stocks_df['P/B'].min()))
        
        # For ROE: higher is better
        stocks_df['ROE_Score'] = 100 * (stocks_df['ROE'] - stocks_df['ROE'].min()) / (stocks_df['ROE'].max() - stocks_df['ROE'].min())
        
        # Composite score (equal weighting)
        stocks_df['Composite_Score'] = (stocks_df['PE_Score'] + stocks_df['PB_Score'] + stocks_df['ROE_Score']) / 3
        
        # Round for display
        stocks_df['P/E'] = stocks_df['P/E'].round(2)
        stocks_df['P/B'] = stocks_df['P/B'].round(2)
        stocks_df['ROE'] = stocks_df['ROE'].round(2)
        stocks_df['Composite_Score'] = stocks_df['Composite_Score'].round(2)
        
        # Group by industry and get top 3 per industry
        top_by_industry = []
        
        for industry in stocks_df['Industry'].unique():
            industry_stocks = stocks_df[stocks_df['Industry'] == industry].copy()
            industry_stocks = industry_stocks.sort_values('Composite_Score', ascending=False).head(3)
            top_by_industry.append(industry_stocks)
        
        # Combine all top stocks
        top_stocks_df = pd.concat(top_by_industry).sort_values(['Industry', 'Composite_Score'], ascending=[True, False])
        
        # Display results
        display_df = top_stocks_df[['Industry', 'Symbol', 'Price', 'P/E', 'P/B', 'ROE', 'Composite_Score', 'Market Cap ($B)']].copy()
        display_df['Price'] = display_df['Price'].round(2)
        display_df['Market Cap ($B)'] = display_df['Market Cap ($B)'].round(2)
        
        print(f"\n=== Top 3 Stocks Per Industry ===")
        print(f"Based on composite score of P/E, P/B, and ROE")
        print(f"Total industries: {len(stocks_df['Industry'].unique())}\n")
        
        display(display_df)
        
        # Summary statistics
        print(f"\n=== Overall Statistics ===")
        print(f"Total stocks analyzed: {len(stocks_df)}")
        print(f"Average P/E: {stocks_df['P/E'].mean():.2f}")
        print(f"Average P/B: {stocks_df['P/B'].mean():.2f}")
        print(f"Average ROE: {stocks_df['ROE'].mean():.2f}%")
        
    else:
        print("No valid stock data collected.")
else:
    print("No universe data available.")

In [ ]:
# Vietnamese Stock Market Scanner
# Note: QuantConnect may not have Vietnamese stock data
# This provides a manual list approach for common Vietnamese stocks

# Common Vietnamese blue-chip stocks (VN30 Index constituents)
vietnamese_tickers = [
    # Banks
    'VCB', 'BID', 'CTG', 'TCB', 'MBB', 'VPB', 'ACB', 'HDB', 'STB', 'TPB',
    # Real Estate
    'VHM', 'VIC', 'NVL', 'VRE', 'PDR', 'KDH',
    # Consumer & Retail
    'VNM', 'MSN', 'MWG',
    # Industry
    'HPG', 'GVR', 'SSI', 'VCI', 'PLX',
    # Utilities
    'GAS', 'POW', 'NT2',
    # Technology
    'FPT'
]

vietnamese_stocks_data = []
failed_tickers = []

print(f"Attempting to scan {len(vietnamese_tickers)} Vietnamese stocks...")
print("Note: Vietnamese stocks may not be available in QuantConnect")
print("If this fails, you'll need to use an external data source\n")

for ticker in vietnamese_tickers:
    try:
        # Try to add Vietnamese stock
        # Format might be: ticker + ".VN" or just ticker
        # Try different formats
        symbol_formats = [ticker, f"{ticker}.VN", f"VN:{ticker}"]
        
        found = False
        for symbol_format in symbol_formats:
            try:
                print(f"Trying {symbol_format}... ", end="")
                equity = qb.add_equity(symbol_format, Resolution.DAILY, market=Market.USA)
                
                # Get price
                history = qb.history(equity.symbol, 5, Resolution.DAILY)
                if history.empty:
                    print("No data")
                    continue
                
                price = history['close'].iloc[-1]
                print(f"✓ Found! Price: {price:.2f}")
                found = True
                
                # Try to get fundamentals
                if equity.symbol in qb.securities:
                    security = qb.securities[equity.symbol]
                    
                    if hasattr(security, 'fundamentals') and security.fundamentals is not None:
                        fundamentals = security.fundamentals
                        
                        pe_ratio = fundamentals.valuation_ratios.pe_ratio
                        pb_ratio = fundamentals.valuation_ratios.pb_ratio
                        roe = fundamentals.operation_ratios.roe.value if fundamentals.operation_ratios.roe else None
                        market_cap = fundamentals.market_cap
                        
                        # Get industry
                        asset_classification = fundamentals.asset_classification
                        industry = asset_classification.morningstar_industry_group_code if asset_classification else "Unknown"
                        
                        if pe_ratio and pb_ratio and roe and pe_ratio > 0 and pb_ratio > 0:
                            vietnamese_stocks_data.append({
                                'Symbol': ticker,
                                'Price': price,
                                'P/E': pe_ratio,
                                'P/B': pb_ratio,
                                'ROE': roe * 100,
                                'Industry': industry,
                                'Market Cap ($M)': market_cap / 1e6 if market_cap else 0
                            })
                
                break  # Found valid format, stop trying
                
            except Exception as e:
                print(f"✗", end=" ")
                continue
        
        if not found:
            print(f"❌ {ticker}: Not available")
            failed_tickers.append(ticker)
            
    except Exception as e:
        print(f"❌ {ticker}: Error - {str(e)}")
        failed_tickers.append(ticker)
        continue

print(f"\n{'='*60}")
print(f"Successfully found: {len(vietnamese_stocks_data)} stocks")
print(f"Failed: {len(failed_tickers)} stocks")

if len(vietnamese_stocks_data) == 0:
    print("\n⚠️ Vietnamese stocks are not available in QuantConnect")
    print("\nAlternative Solutions:")
    print("1. Use external Vietnamese stock data APIs:")
    print("   - SSI Securities API")
    print("   - TCBS API")
    print("   - VND Direct API")
    print("   - cafe.vn data")
    print("\n2. Manual data entry approach:")
    print("   - Collect data from Vietnamese stock websites")
    print("   - Create a CSV with ticker, P/E, P/B, ROE, industry")
    print("   - Load and analyze in pandas")
    print("\n3. See the manual CSV example below...")
    
else:
    # Display results if we found any stocks
    vn_df = pd.DataFrame(vietnamese_stocks_data)
    
    # Calculate composite score
    vn_df['PE_Score'] = 100 * (1 - (vn_df['P/E'] - vn_df['P/E'].min()) / (vn_df['P/E'].max() - vn_df['P/E'].min()))
    vn_df['PB_Score'] = 100 * (1 - (vn_df['P/B'] - vn_df['P/B'].min()) / (vn_df['P/B'].max() - vn_df['P/B'].min()))
    vn_df['ROE_Score'] = 100 * (vn_df['ROE'] - vn_df['ROE'].min()) / (vn_df['ROE'].max() - vn_df['ROE'].min())
    vn_df['Composite_Score'] = (vn_df['PE_Score'] + vn_df['PB_Score'] + vn_df['ROE_Score']) / 3
    
    # Round for display
    vn_df['P/E'] = vn_df['P/E'].round(2)
    vn_df['P/B'] = vn_df['P/B'].round(2)
    vn_df['ROE'] = vn_df['ROE'].round(2)
    vn_df['Composite_Score'] = vn_df['Composite_Score'].round(2)
    
    # Group by industry and get top 3
    top_vn_stocks = []
    for industry in vn_df['Industry'].unique():
        industry_stocks = vn_df[vn_df['Industry'] == industry].sort_values('Composite_Score', ascending=False).head(3)
        top_vn_stocks.append(industry_stocks)
    
    result_df = pd.concat(top_vn_stocks).sort_values(['Industry', 'Composite_Score'], ascending=[True, False])
    
    print("\n=== Top Vietnamese Stocks by Industry ===")
    display(result_df[['Industry', 'Symbol', 'Price', 'P/E', 'P/B', 'ROE', 'Composite_Score']])

In [ ]:
# Manual Vietnamese Stock Data Approach
# Example data - replace with real data from Vietnamese stock sources

# Example: Create a DataFrame with Vietnamese stock data
# Data sources: SSI, TCBS, VND Direct, cafe.vn, vietstock.vn

vietnamese_data = {
    'Symbol': ['VCB', 'BID', 'CTG', 'VHM', 'VIC', 'VNM', 'HPG', 'FPT', 'GAS', 'MSN', 
               'TCB', 'MBB', 'VPB', 'ACB', 'MWG', 'GVR', 'SSI', 'PLX', 'POW', 'HDB'],
    'Price': [92500, 47800, 35500, 45200, 38900, 87500, 25800, 121000, 102500, 158000,
              55300, 26500, 22100, 26800, 61500, 24700, 31200, 38600, 11200, 28400],
    'P/E': [11.2, 8.5, 7.8, 15.3, 18.2, 14.5, 9.2, 16.8, 12.4, 22.5,
            9.8, 8.9, 10.5, 11.3, 13.2, 8.7, 12.1, 10.8, 7.5, 9.4],
    'P/B': [2.8, 1.9, 1.5, 2.1, 3.2, 6.5, 1.2, 4.3, 2.9, 3.8,
            1.8, 1.6, 1.4, 1.7, 2.5, 0.9, 2.2, 1.3, 0.8, 1.5],
    'ROE': [24.8, 22.5, 19.2, 13.7, 17.6, 44.8, 13.0, 25.6, 23.4, 16.9,
            18.3, 18.0, 13.3, 15.0, 19.0, 10.3, 18.2, 12.0, 10.6, 16.0],
    'Industry': ['Banking', 'Banking', 'Banking', 'Real Estate', 'Real Estate', 'Consumer',
                 'Industrial', 'Technology', 'Utilities', 'Consumer',
                 'Banking', 'Banking', 'Banking', 'Banking', 'Retail', 'Industrial',
                 'Financial Services', 'Utilities', 'Utilities', 'Banking'],
    'Market_Cap_VND_Billion': [460000, 235000, 178000, 210000, 185000, 168000,
                                120000, 155000, 152000, 105000,
                                98000, 92000, 88000, 85000, 68000, 65000,
                                45000, 95000, 42000, 72000]
}

vn_stocks = pd.DataFrame(vietnamese_data)

print("Vietnamese Stock Scanner (Manual Data)")
print("=" * 60)
print(f"Total stocks: {len(vn_stocks)}")
print(f"Industries: {vn_stocks['Industry'].nunique()}")
print("\nNote: Update the data with current values from Vietnamese stock sources")
print("=" * 60)

# Calculate composite scores
vn_stocks['PE_Score'] = 100 * (1 - (vn_stocks['P/E'] - vn_stocks['P/E'].min()) / (vn_stocks['P/E'].max() - vn_stocks['P/E'].min()))
vn_stocks['PB_Score'] = 100 * (1 - (vn_stocks['P/B'] - vn_stocks['P/B'].min()) / (vn_stocks['P/B'].max() - vn_stocks['P/B'].min()))
vn_stocks['ROE_Score'] = 100 * (vn_stocks['ROE'] - vn_stocks['ROE'].min()) / (vn_stocks['ROE'].max() - vn_stocks['ROE'].min())

# Composite score (equal weighting)
vn_stocks['Composite_Score'] = (vn_stocks['PE_Score'] + vn_stocks['PB_Score'] + vn_stocks['ROE_Score']) / 3
vn_stocks['Composite_Score'] = vn_stocks['Composite_Score'].round(2)

# Rank within each industry
vn_stocks['Industry_Rank'] = vn_stocks.groupby('Industry')['Composite_Score'].rank(ascending=False, method='dense')

# Get top 3 per industry
top_3_per_industry = vn_stocks[vn_stocks['Industry_Rank'] <= 3].copy()
top_3_per_industry = top_3_per_industry.sort_values(['Industry', 'Composite_Score'], ascending=[True, False])

# Display results
display_cols = ['Industry', 'Symbol', 'Price', 'P/E', 'P/B', 'ROE', 'Composite_Score', 'Industry_Rank']
result = top_3_per_industry[display_cols].copy()

print("\n=== Top 3 Vietnamese Stocks Per Industry ===\n")
display(result)

# Overall top 10
print("\n=== Overall Top 10 Vietnamese Value Stocks ===\n")
top_10 = vn_stocks.nlargest(10, 'Composite_Score')[['Symbol', 'Industry', 'Price', 'P/E', 'P/B', 'ROE', 'Composite_Score']]
display(top_10)

# Statistics by industry
print("\n=== Industry Statistics ===\n")
industry_stats = vn_stocks.groupby('Industry').agg({
    'P/E': 'mean',
    'P/B': 'mean',
    'ROE': 'mean',
    'Composite_Score': 'mean',
    'Symbol': 'count'
}).round(2)
industry_stats.columns = ['Avg P/E', 'Avg P/B', 'Avg ROE', 'Avg Score', 'Count']
display(industry_stats.sort_values('Avg Score', ascending=False))

# Export option
print("\n💾 Export Results:")
print("Uncomment the lines below to save results to CSV")
print("# top_3_per_industry.to_csv('vn_top_stocks_by_industry.csv', index=False)")
print("# top_10.to_csv('vn_top_10_stocks.csv', index=False)")

## Alternative: Manual Vietnamese Stock Data
If QuantConnect doesn't have Vietnamese stocks, use this manual approach with CSV data

## Vietnamese Stock Market Scanner
Scan Vietnamese stocks (HOSE/HNX) with P/E, P/B, ROE ranking by industry

## Industry-Based Ranking Scanner
Rank stocks by P/E, P/B, and ROE, showing top 3 per industry

## Relaxed Screener (More Realistic Thresholds)
Since < 1M shares is extremely rare, here's a version with more realistic thresholds

In [ ]:
# Optional: Export to CSV
if screened_stocks:
    results_df.to_csv('screened_stocks.csv', index=False)
    print("Results saved to screened_stocks.csv")

## Alternative: Screen with Broader Universe
## If the S&P 500 is too restrictive, you can expand to other universes

## Alternative Approach: Use a Manual Symbol List
If the ETF universe approach has issues, you can screen a manual list of symbols

In [ ]:
# Alternative: Screen a manual list of small-cap/mid-cap symbols
# This approach is useful if universe_history has issues

# Example list of diverse symbols (you can expand this)
manual_symbols = ['AAPL', 'MSFT', 'F', 'BAC', 'T', 'INTC', 'CSCO', 'PFE', 'KO', 
                  'WFC', 'C', 'XOM', 'CVX', 'GE', 'IBM', 'ORCL', 'MRK', 'VZ',
                  'GM', 'HPQ', 'AMD', 'QCOM', 'MU', 'TXN', 'LMT', 'RTX']

screened_manual = []

print(f"Screening {len(manual_symbols)} manually selected symbols...")

for ticker in manual_symbols:
    try:
        # Add the equity
        symbol = qb.add_equity(ticker, Resolution.DAILY).symbol
        
        # Get price
        history = qb.history(symbol, 1, Resolution.DAILY)
        if history.empty:
            continue
        price = history['close'].iloc[-1]
        
        # Check price range
        if price < 10 or price > 100:
            continue
        
        # Get the security object to access fundamentals
        security = qb.securities[symbol]
        if not hasattr(security, 'fundamentals') or security.fundamentals is None:
            continue
        
        fundamentals = security.fundamentals
        
        # Get valuation ratios
        pe_ratio = fundamentals.valuation_ratios.pe_ratio
        pb_ratio = fundamentals.valuation_ratios.pb_ratio
        market_cap = fundamentals.market_cap
        
        if market_cap <= 0 or price <= 0:
            continue
            
        shares_outstanding = market_cap / price
        
        # Apply screening criteria
        if (pe_ratio is not None and pe_ratio > 0 and pe_ratio < 10 and
            pb_ratio is not None and pb_ratio > 0 and pb_ratio < 1.0 and
            shares_outstanding > 0 and shares_outstanding < 1_000_000):
            
            screened_manual.append({
                'Symbol': ticker,
                'Price': round(price, 2),
                'P/E Ratio': round(pe_ratio, 2),
                'P/B Ratio': round(pb_ratio, 2),
                'Shares (M)': round(shares_outstanding / 1_000_000, 2),
                'Market Cap ($M)': round(market_cap / 1_000_000, 2)
            })
    except Exception as e:
        # print(f"Error with {ticker}: {str(e)}")
        continue

print(f"Found {len(screened_manual)} stocks matching criteria from manual list")

if screened_manual:
    manual_df = pd.DataFrame(screened_manual)
    display(manual_df.sort_values('P/E Ratio'))

In [ ]:
# Screen across multiple ETF universes for better coverage
etf_tickers = ['SPY', 'IWM', 'QQQ', 'DIA']  # S&P 500, Russell 2000, Nasdaq 100, Dow 30

all_universe_symbols = set()
for etf in etf_tickers:
    try:
        universe_history = qb.universe_history(
            qb.universe.etf(etf),
            screening_date - timedelta(1),
            screening_date
        )
        if not universe_history.empty:
            constituents = [fundamental.symbol for fundamental in universe_history.iloc[0]]
            all_universe_symbols.update(constituents)
            print(f"Added {len(constituents)} symbols from {etf}")
    except Exception as e:
        print(f"Error loading {etf}: {str(e)}")

print(f"\nTotal unique symbols: {len(all_universe_symbols)}")

# Now run the screening logic on this expanded universe
# Replace 'spy_constituents' with 'all_universe_symbols' in the screening cell above